# 02. 구조화된 데이터 추출 (Structured Extraction) 🏪

## 학습 목표
- LLM 응답에서 구조화된 데이터를 정확하게 추출하는 기법 학습
- JSON Mode, Function Calling 개념 이해
- Pydantic으로 스키마 정의 및 검증
- 출력 파싱 전략 (regex, json.loads, retry)
- ai-ipsonum 매장 정보 구조화 추출 실습

## ai-ipsonum 연계 🏪
- AI 응답 텍스트 → 구조화된 매장 데이터 추출
- rule-based 파싱 vs LLM 기반 추출 비교

---

In [ ]:
import json
import re
from dataclasses import dataclass, field, asdict
from typing import Optional

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import numpy as np
import pandas as pd

# Pydantic (Colab에서 기본 설치되어 있음)
try:
    from pydantic import BaseModel, Field, validator, ValidationError
    PYDANTIC_AVAILABLE = True
    print("Pydantic 사용 가능")
except ImportError:
    PYDANTIC_AVAILABLE = False
    print("Pydantic 미설치 - dataclass로 대체합니다")
    print("\uc124\uce58: !pip install pydantic")

## 1. LLM에서 구조화된 출력 얻기: 왜 어려운가

LLM은 기본적으로 **자연어 텍스트**를 생성한다. 구조화된 출력(JSON, XML 등)은 LLM에게 자연스러운 형식이 아니다.

### 주요 어려움

| 문제 | 예시 |
|------|------|
| **형식 위반** | JSON에 설명 텍스트 섮여 출력 |
| **필드 누락** | 요청한 필드 중 일부 누락 |
| **타입 불일치** | 숫자를 문자열로 출력 ("3" vs 3) |
| **없는 필드 추가** | 요청하지 않은 필드 임의 추가 |
| **Hallucination** | 존재하지 않는 정보 생성 |

In [ ]:
# LLM 응답의 다양한 형식 문제 예시

# 예시 1: JSON 앞뒤에 설명 섮여 출력
bad_response_1 = """\ub124, \uc131\uc218\ub3d9 \uce74\ud398 \ub9ac\uc2a4\ud2b8\ub97c \uc815\ub9ac\ud588\uc2b5\ub2c8\ub2e4:

[{"name": "\ube14\ub8e8\ubcf4\ud2c0 \uc131\uc218\uc810", "rank": 1}]

\uc774 \uc911\uc5d0\uc11c \ube14\ub8e8\ubcf4\ud2c0\uc774 \uac00\uc7a5 \uc720\uba85\ud569\ub2c8\ub2e4!"""

# 예시 2: 필드 누락
bad_response_2 = '[{"name": "\ube14\ub8e8\ubcf4\ud2c0 \uc131\uc218\uc810"}]'  # rank, category 누락

# 예시 3: 타입 불일치
bad_response_3 = '[{"name": "\ube14\ub8e8\ubcf4\ud2c0 \uc131\uc218\uc810", "rank": "1\uc704", "category": "\uce74\ud398"}]'

print("=== 문제적 응답 예시 ===")
print("\n[1] JSON 앞뒤 설명 섮임:")
print(bad_response_1)
print("\n[2] 필드 누락:")
print(bad_response_2)
print("\n[3] 타입 불일치 (rank가 '1위' 문자열):")
print(bad_response_3)

---
## 2. JSON Mode: OpenAI의 response_format

OpenAI API는 `response_format={"type": "json_object"}`를 제공하여 **반드시 유효한 JSON**으로 출력하도록 강제할 수 있다.

### JSON Mode 사용법 (개념)

```python
# OpenAI API 예시 (\uc2e4\uc81c \ud638\ucd9c \ucf54\ub4dc)
response = client.chat.completions.create(
    model="gpt-4o",
    response_format={"type": "json_object"},  # JSON Mode
    messages=[
        {"role": "system", "content": "JSON\uc73c\ub85c \uc751\ub2f5\ud558\uc138\uc694."},
        {"role": "user", "content": query}
    ]
)
```

### 제약과 활용

| 항목 | 내용 |
|------|------|
| **장점** | JSON 유효성 보장, 파싱 실패 없음 |
| **단점** | 스키마(키 이름, 타입)는 보장하지 않음 |
| **지원 모델** | GPT-4o, GPT-4 Turbo 등 (GPT-3.5는 제한적) |
| **비용** | 일반 호출과 동일 |

In [ ]:
# JSON Mode 시뮬레이션

def mock_llm_json_mode(prompt: str, json_mode: bool = False) -> str:
    """JSON Mode 유무에 따른 응답 차이 시뮬레이션"""
    if json_mode:
        # JSON Mode ON: 유효한 JSON만 출력
        return json.dumps({
            "stores": [
                {"name": "블루보틀 성수점", "rank": 1, "category": "카페"},
                {"name": "스타벅스 종로점", "rank": 2, "category": "카페"},
            ]
        }, ensure_ascii=False, indent=2)
    else:
        # JSON Mode OFF: 설명과 섮여 출력
        return """\uc131\uc218\ub3d9 \uc778\uae30 \uce74\ud398\ub97c \uc54c\ub824\ub4dc\ub9ac\uaca0\uc2b5\ub2c8\ub2e4.

{"stores": [{"name": "\ube14\ub8e8\ubcf4\ud2c0 \uc131\uc218\uc810", "rank": 1}]}

\uc704 \ub9e4\uc7a5\ub4e4\uc774 \uc778\uae30\uac00 \ub9ce\uc2b5\ub2c8\ub2e4!"""


# 비교
print("=== JSON Mode OFF ===")
resp_off = mock_llm_json_mode("성수동 카페", json_mode=False)
print(resp_off)
try:
    json.loads(resp_off)
    print("\u2713 JSON 파싱 성공")
except json.JSONDecodeError as e:
    print(f"\u2717 JSON 파싱 실패: {e}")

print("\n=== JSON Mode ON ===")
resp_on = mock_llm_json_mode("성수동 카페", json_mode=True)
print(resp_on)
try:
    parsed = json.loads(resp_on)
    print(f"\u2713 JSON 파싱 성공: {len(parsed['stores'])}개 매장")
except json.JSONDecodeError as e:
    print(f"\u2717 JSON 파싱 실패: {e}")

---
## 3. Function Calling / Tool Use: 구조화된 입출력 정의

Function Calling은 LLM에게 **사용 가능한 함수의 스키마**를 알려주는 방식.
LLM이 직접 함수를 실행하는 것이 아니라, **어떤 함수를 어떤 인자로 호출해야 하는지** 결정한다.

### 흐름

```
1. 개발자: 함수 스키마 정의 (\uc774\ub984, \ud30c\ub77c\ubbf8\ud130, \ud0c0\uc785)
2. LLM: \uc0ac\uc6a9\uc790 \uc785\ub825\uc744 \ubcf4\uace0 \ud568\uc218 \ud638\ucd9c \uacb0\uc815
3. \uac1c\ubc1c\uc790: \uc2e4\uc81c \ud568\uc218 \uc2e4\ud589
4. LLM: \ud568\uc218 \uacb0\uacfc\ub97c \ubc14\ud0d5\uc73c\ub85c \uc751\ub2f5 \uc0dd\uc131
```

In [ ]:
# Function Calling 스키마 정의 예시 (OpenAI 형식)

tool_schema = {
    "type": "function",
    "function": {
        "name": "extract_store_info",
        "description": "AI 응답에서 매장 정보를 구조화하여 추출합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "stores": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string", "description": "매장명"},
                            "rank": {"type": "integer", "description": "순위 (1부터)"},
                            "category": {"type": "string", "description": "카테고리 (카페/맛집/바 등)"},
                            "reason": {"type": "string", "description": "추천 이유"}
                        },
                        "required": ["name", "rank", "category"]
                    }
                }
            },
            "required": ["stores"]
        }
    }
}

print("=== Function Calling Schema ===")
print(json.dumps(tool_schema, indent=2, ensure_ascii=False))

# LLM이 반환하는 함수 호출 예시
mock_function_call = {
    "name": "extract_store_info",
    "arguments": json.dumps({
        "stores": [
            {"name": "블루보틀 성수점", "rank": 1, "category": "카페", "reason": "스페셜티 드립 커피"},
            {"name": "스타벅스 종로점", "rank": 2, "category": "카페", "reason": "넓은 좌석"}
        ]
    }, ensure_ascii=False)
}

print("\n=== LLM이 반환하는 Function Call ===")
print(json.dumps(mock_function_call, indent=2, ensure_ascii=False))

# 파싱 후 실행
args = json.loads(mock_function_call["arguments"])
print(f"\n\u2192 추출된 매장 수: {len(args['stores'])}")
for store in args["stores"]:
    print(f"  #{store['rank']} {store['name']} [{store['category']}]")

---
## 4. Pydantic으로 스키마 정의: BaseModel로 출력 형식 강제

Pydantic은 Python의 데이터 검증 라이브러리.
LLM 출력을 Pydantic 모델로 파싱하면 **타입 검증, 필드 누락 검사, 값 범위 검증**을 자동으로 할 수 있다.

```python
class StoreInfo(BaseModel):
    name: str
    rank: int = Field(ge=1, le=10)  # 1~10 범위
    category: str
    reason: Optional[str] = None
```

In [ ]:
# Pydantic 모델 정의 및 검증

if PYDANTIC_AVAILABLE:
    class StoreInfo(BaseModel):
        name: str
        rank: int = Field(ge=1, le=10, description="순위 (1~10)")
        category: str = Field(description="카테고리 (카페/맛집/바 등)")
        reason: Optional[str] = Field(None, description="추천 이유")
    
    class StoreExtractionResult(BaseModel):
        stores: list[StoreInfo]
        total_count: int = Field(ge=0)
        query_area: str
    
    # 정상 데이터 검증
    print("=== 정상 데이터 ===")
    valid_data = {
        "stores": [
            {"name": "블루보틀 성수점", "rank": 1, "category": "카페", "reason": "스페셜티 커피"},
            {"name": "스타벅스 종로점", "rank": 2, "category": "카페"}
        ],
        "total_count": 2,
        "query_area": "성수동"
    }
    result = StoreExtractionResult(**valid_data)
    print(f"\u2713 검증 성공: {len(result.stores)}개 매장")
    for s in result.stores:
        print(f"  #{s.rank} {s.name} [{s.category}] - {s.reason or 'N/A'}")
    
    # 잘못된 데이터 검증
    print("\n=== 잘못된 데이터 ===")
    invalid_cases = [
        {"name": "블루보틀", "rank": 0, "category": "카페"},   # rank < 1
        {"name": "스타벅스", "rank": 15, "category": "카페"},  # rank > 10
        {"name": "", "rank": 1, "category": "카페"},           # 빈 name
    ]
    for case in invalid_cases:
        try:
            StoreInfo(**case)
            print(f"  \u2713 {case} - 통과")
        except ValidationError as e:
            print(f"  \u2717 {case}")
            for err in e.errors():
                print(f"    → {err['loc']}: {err['msg']}")

else:
    # Pydantic 없을 때 dataclass로 대체
    @dataclass
    class StoreInfo:
        name: str
        rank: int
        category: str
        reason: Optional[str] = None
    
    store = StoreInfo(name="블루보틀 성수점", rank=1, category="카페")
    print(f"Store: {store}")
    print("→ dataclass는 타입 검증을 자동으로 하지 않음. 직접 검증 필요.")

In [ ]:
# Pydantic 모델에서 JSON Schema 자동 생성

if PYDANTIC_AVAILABLE:
    schema = StoreInfo.model_json_schema()
    print("=== Pydantic → JSON Schema ===")
    print(json.dumps(schema, indent=2, ensure_ascii=False))
    print("\n\u2192 이 스키마를 LLM 프롬프트나 Function Calling에 그대로 사용 가능")
else:
    print("→ Pydantic이 없어 JSON Schema 자동 생성 불가. 수동 정의 필요.")

---
## 5. 출력 파싱 전략: regex, json.loads, retry on failure

LLM 출력이 항상 완벽한 JSON인 것은 아니다. 다양한 파싱 전략이 필요.

### 파싱 전략 우선순위

1. **json.loads**: 단순 파싱 시도
2. **JSON 추출**: 텍스트에서 `[...]` 또는 `{...}` 부분만 추출
3. **regex 파싱**: 패턴 매칭으로 구조화
4. **LLM 재호출**: 파싱 실패 시 출력을 다시 JSON으로 변환 요청

In [ ]:
# 다양한 파싱 전략 구현

def extract_json_from_text(text: str) -> Optional[Any]:
    """텍스트에서 JSON을 추출하는 함수 (\ub2e4양한 전\ub7b5 적\uc6a9)"""
    
    # 전략 1: 직접 파싱
    try:
        return json.loads(text), "direct_parse"
    except json.JSONDecodeError:
        pass
    
    # 전략 2: JSON 블록 추출 (```json ... ``` 패턴)
    json_block = re.search(r'```json\s*\n(.*?)\n\s*```', text, re.DOTALL)
    if json_block:
        try:
            return json.loads(json_block.group(1)), "json_block"
        except json.JSONDecodeError:
            pass
    
    # 전략 3: [ ... ] 또는 { ... } 추출
    for pattern in [r'(\[.*\])', r'(\{.*\})']:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1)), "regex_extract"
            except json.JSONDecodeError:
                pass
    
    # 전략 4: 라인별 파싱 ("1. 매장명 - 설명" 패턴)
    line_pattern = r'(\d+)\.\s+(.+?)\s*[-:]\s*(.+?)$'
    matches = re.findall(line_pattern, text, re.MULTILINE)
    if matches:
        items = [{"rank": int(m[0]), "name": m[1].strip(), "description": m[2].strip()} for m in matches]
        return items, "line_parse"
    
    return None, "failed"


# 다양한 LLM 응답 테스트
test_responses = [
    # Case 1: 깨끗한 JSON
    '[{"name": "\ube14\ub8e8\ubcf4\ud2c0", "rank": 1}]',
    
    # Case 2: 설명 + JSON 섮임
    '\uc131\uc218\ub3d9 \uce74\ud398 \ub9ac\uc2a4\ud2b8:\n[{"name": "\ube14\ub8e8\ubcf4\ud2c0", "rank": 1}]\n\uc774\uc0c1\uc785\ub2c8\ub2e4.',
    
    # Case 3: 코드 블록
    '```json\n[{"name": "\ube14\ub8e8\ubcf4\ud2c0", "rank": 1}]\n```',
    
    # Case 4: 순수 텍스트
    '1. \ube14\ub8e8\ubcf4\ud2c0 - \uc2a4\ud398\uc15c\ud2f0 \ucee4\ud53c\n2. \uc2a4\ud0c0\ubc85\uc2a4 - \uc811\uadfc\uc131 \uc88b\uc74c',
]

print("=== 파싱 전략 테스트 ===")
for i, resp in enumerate(test_responses, 1):
    result, strategy = extract_json_from_text(resp)
    status = "\u2713" if result else "\u2717"
    print(f"\nCase {i} [{strategy}] {status}")
    print(f"  \uc785\ub825: {resp[:50]}..." if len(resp) > 50 else f"  \uc785\ub825: {resp}")
    print(f"  \uacb0\uacfc: {result}")

In [ ]:
# Retry 로직 구현

def parse_with_retry(text: str, max_retries: int = 3) -> dict:
    """
    파싱 실패 시 재시도하는 로직.
    실제로는 LLM을 다시 호출하지만, 여기서는 점진적 구조 복구를 시뮬레이션.
    """
    attempts = []
    
    for attempt in range(max_retries):
        result, strategy = extract_json_from_text(text)
        
        if result is not None:
            attempts.append({"attempt": attempt + 1, "strategy": strategy, "success": True})
            return {"data": result, "attempts": attempts}
        
        attempts.append({"attempt": attempt + 1, "strategy": "failed", "success": False})
        
        # 재시도: 텍스트 정제 (실제로는 LLM 재호출)
        text = text.strip()
        # 마크다운 / 텍스트 제거 시도
        text = re.sub(r'^[^\[\{]+', '', text)
        text = re.sub(r'[^\]\}]+$', '', text)
    
    return {"data": None, "attempts": attempts}


# 테스트
messy_text = """네, \uc5ec\uae30 \uacb0\uacfc\uc785\ub2c8\ub2e4.
[{"name": "\ube14\ub8e8\ubcf4\ud2c0", "rank": 1, "category": "\uce74\ud398"}]
\ub3c4\uc6c0\uc774 \ub418\uc168\uae30\ub97c!"""

result = parse_with_retry(messy_text)
print("=== Retry 파싱 결과 ===")
print(f"\uc131\uacf5: {result['data'] is not None}")
print(f"\uc2dc\ub3c4 \ud69f\uc218: {len(result['attempts'])}")
for a in result["attempts"]:
    print(f"  Attempt {a['attempt']}: {a['strategy']} ({'\u2713' if a['success'] else '\u2717'})")
if result["data"]:
    print(f"\ub370\uc774\ud130: {result['data']}")

---
## 6. 🏪 매장 정보 구조화 추출

ai-ipsonum에서 실제로 필요한 작업:  
AI 응답 텍스트 → `{"name": "블루보틀", "rank": 1, "category": "카페"}` 변환

### rule-based 파싱 vs LLM 기반 추출

| 방법 | 장점 | 단점 |
|------|------|------|
| **Rule-based** | 빠르고 예측 가능, 비용 없음 | 다양한 형식 대응 어려움 |
| **LLM 기반** | 유연한 형식 대응 | 비용 발생, 지연 |

In [ ]:
# --- 샘플 AI 응답 데이터 (3개 엔진 응답 시뮬레이션) ---

sample_ai_responses = {
    "engine_a": """\uc131\uc218\ub3d9 \uc778\uae30 \uce74\ud398 TOP 5

1. \ube14\ub8e8\ubcf4\ud2c0 \uc131\uc218\uc810 - \uc2a4\ud398\uc15c\ud2f0 \ub4dc\ub9bd \ucee4\ud53c\ub85c \uc720\uba85\ud55c \uce74\ud398
2. \uc2a4\ud0c0\ubc85\uc2a4 \uc885\ub85c\uc810 - \ub113\uc740 \uc88c\uc11d\uacfc \ud3b8\ub9ac\ud55c \uc811\uadfc\uc131
3. \ucee8\ud14c\uc774\ub108 \uc131\uc218\ub3d9 - \ub8e8\ud504\ud0d1\uc774 \uc788\ub294 \ub85c\uc2a4\ud130\ub9ac
4. \uc194\uae38\uccb4 - \ub2e8\uccb4 \ubaa8\uc784\uc5d0 \uc88b\uc740 \ub113\uc740 \uacf5\uac04
5. \ud50c\ub9bf\ud654\uc774\ud2b8 - \ubc14\ub2e4 \ubc14\ub9ac\uc2a4\ud0c0 \uc2a4\ud0c0\uc77c \uce74\ud398""",

    "engine_b": """[{"name": "\ube14\ub8e8\ubcf4\ud2c0 \uc131\uc218\uc810", "rank": 1, "category": "\uce74\ud398", "reason": "\uc2a4\ud398\uc15c\ud2f0 \ucee4\ud53c"}, {"name": "\uc2a4\ud0c0\ubc85\uc2a4 \uc885\ub85c\uc810", "rank": 2, "category": "\uce74\ud398", "reason": "\uc811\uadfc\uc131"}, {"name": "\ucee8\ud14c\uc774\ub108", "rank": 3, "category": "\uce74\ud398", "reason": "\ub8e8\ud504\ud0d1"}, {"name": "\uc194\uae38\uccb4", "rank": 4, "category": "\uce74\ud398", "reason": "\ub2e8\uccb4\uc11d"}]""",

    "engine_c": """\ub124, \uc131\uc218\ub3d9 \uce74\ud398\ub97c \ucd94\ucc9c\ud574 \ub4dc\ub9ac\uaca0\uc2b5\ub2c8\ub2e4.

```json
[
  {"name": "\ube14\ub8e8\ubcf4\ud2c0 \uc131\uc218\uc810", "rank": 1, "category": "\uce74\ud398"},
  {"name": "\ucee8\ud14c\uc774\ub108 \uc131\uc218\ub3d9", "rank": 2, "category": "\uce74\ud398"},
  {"name": "\uc2a4\ud0c0\ubc85\uc2a4 \uc885\ub85c\uc810", "rank": 3, "category": "\uce74\ud398"}
]
```

\uc774 \uc911\uc5d0\uc11c \ube14\ub8e8\ubcf4\ud2c0\uc774 \uac00\uc7a5 \uc778\uae30\uac00 \ub9ce\uc2b5\ub2c8\ub2e4!"""
}

for engine, response in sample_ai_responses.items():
    print(f"=== {engine} \uc751\ub2f5 \ubbf8\ub9ac\ubcf4\uae30 ===")
    print(response[:80] + "..." if len(response) > 80 else response)
    print()

In [ ]:
# --- Rule-based 파싱 ---

def rule_based_extract(text: str) -> list[dict]:
    """정규식 기반으로 매장 정보 추출"""
    stores = []
    
    # 패턴 1: JSON 배열 직접 파싱 시도
    result, strategy = extract_json_from_text(text)
    if result and isinstance(result, list):
        for item in result:
            if isinstance(item, dict) and "name" in item:
                stores.append({
                    "name": item.get("name", ""),
                    "rank": item.get("rank", 0),
                    "category": item.get("category", "미정")
                })
        return stores
    
    # 패턴 2: "1. 매장명 - 설명" 형식
    pattern = r'(\d+)\.\s+(.+?)\s*[-\u2013]\s*(.+?)$'
    matches = re.findall(pattern, text, re.MULTILINE)
    for rank, name, description in matches:
        stores.append({
            "name": name.strip(),
            "rank": int(rank),
            "category": "미정"  # 텍스트에서 카테고리 추출 어려움
        })
    
    return stores


# 각 엔진 응답에 rule-based 파싱 적용
print("=== Rule-based 파싱 결과 ===")
all_extracted = {}
for engine, response in sample_ai_responses.items():
    extracted = rule_based_extract(response)
    all_extracted[engine] = extracted
    print(f"\n{engine}: {len(extracted)}개 매장 추출")
    for s in extracted:
        print(f"  #{s['rank']} {s['name']} [{s['category']}]")

In [ ]:
# --- LLM 기반 추출 (Mock) ---

def mock_llm_extract(text: str) -> list[dict]:
    """
    LLM을 활용한 구조화 추출 시뮬레이션.
    실제로는 LLM에게 "이 텍스트에서 매장 정보를 JSON으로 추출해줘"라고 요청.
    """
    # 텍스트에서 키워드 기반으로 매장 인식 (실제 LLM이 하는 작업을 시뮬레이션)
    known_stores = {
        "블루보틀": {"full_name": "블루보틀 성수점", "category": "카페"},
        "스타벅스": {"full_name": "스타벅스 종로점", "category": "카페"},
        "컨테이너": {"full_name": "컨테이너 성수동", "category": "카페"},
        "솔길체": {"full_name": "솔길체", "category": "카페"},
        "플릿화이트": {"full_name": "플릿화이트", "category": "카페"},
    }
    
    results = []
    rank = 1
    for keyword, info in known_stores.items():
        if keyword in text:
            results.append({
                "name": info["full_name"],
                "rank": rank,
                "category": info["category"]
            })
            rank += 1
    
    return results


# LLM 기반 추출 결과
print("=== LLM 기반 추출 결과 ===")
llm_extracted = {}
for engine, response in sample_ai_responses.items():
    extracted = mock_llm_extract(response)
    llm_extracted[engine] = extracted
    print(f"\n{engine}: {len(extracted)}개 매장 추출")
    for s in extracted:
        print(f"  #{s['rank']} {s['name']} [{s['category']}]")

In [ ]:
# --- Rule-based vs LLM 기반 추출 비교 ---

ground_truth_names = {"블루보틀 성수점", "스타벅스 종로점", "컨테이너 성수동", "솔길체", "플릿화이트"}

def evaluate_extraction_method(extracted_by_engine: dict, gt_names: set) -> dict:
    """ 추출 결과 평가"""
    total_precision = []
    total_recall = []
    has_category = []
    
    for engine, stores in extracted_by_engine.items():
        extracted_names = {s["name"] for s in stores}
        tp = len(extracted_names & gt_names)
        precision = tp / len(extracted_names) if extracted_names else 0
        recall = tp / len(gt_names) if gt_names else 0
        total_precision.append(precision)
        total_recall.append(recall)
        has_category.append(sum(1 for s in stores if s["category"] != "미정") / len(stores) if stores else 0)
    
    return {
        "avg_precision": np.mean(total_precision),
        "avg_recall": np.mean(total_recall),
        "category_fill_rate": np.mean(has_category)
    }

eval_rule = evaluate_extraction_method(all_extracted, ground_truth_names)
eval_llm = evaluate_extraction_method(llm_extracted, ground_truth_names)

comparison = pd.DataFrame({
    "Metric": ["Avg Precision", "Avg Recall", "Category Fill Rate"],
    "Rule-based": [eval_rule["avg_precision"], eval_rule["avg_recall"], eval_rule["category_fill_rate"]],
    "LLM-based": [eval_llm["avg_precision"], eval_llm["avg_recall"], eval_llm["category_fill_rate"]]
})
print(comparison.to_string(index=False))

# 시각화
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(comparison))
width = 0.35
bars1 = ax.bar(x - width/2, comparison["Rule-based"], width, label="Rule-based", color="#ff6b6b")
bars2 = ax.bar(x + width/2, comparison["LLM-based"], width, label="LLM-based", color="#4ecdc4")

ax.set_ylabel("Score")
ax.set_title("Rule-based vs LLM-based Extraction")
ax.set_xticks(x)
ax.set_xticklabels(comparison["Metric"])
ax.legend()
ax.set_ylim(0, 1.2)
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: Pydantic 모델 정의

맛집 정보를 위한 Pydantic 모델을 정의하세요.  
요구사항:
- `name` (str): 매장명
- `rank` (int): 1~20 범위
- `category` (str): "한식" | "일식" | "양식" | "중식" | "기타"
- `price_range` (Optional[str]): "저가" | "중가" | "고가" | None
- `rating` (Optional[float]): 0.0~5.0

In [ ]:
# TODO: Pydantic 모델을 정의하세요
# 요구사항:
# 1. RestaurantInfo BaseModel 정의
# 2. category는 허용된 값만 허용 (validator 사용)
# 3. 정상/비정상 데이터 검증 테스트

# class RestaurantInfo(BaseModel):
#     TODO


### 연습 2: 새로운 AI 응답 파싱

아래 AI 응답에서 매장 정보를 추출하는 함수를 작성하세요.

In [ ]:
# TODO: 아래 AI 응답에서 매장 정보를 추출하는 함수를 작성하세요

new_ai_response = """
\uac15\ub0a8\uc5ed \uadfc\ucc98 \ub9db\uc9d1\uc744 \uc54c\ub824\ub4dc\ub9b4\uac8c\uc694!

\u2b50 1\uc704: \uc720\uc721\uc758 \ub2ec\uc778 \uac15\ub0a8\uc810 (\ud55c\uc2dd) - \uc0dd\uace0\uae30 \uc804\ubb38\uc810, \uc608\uc57d \ud544\uc218
\u2b50 2\uc704: \uc624\ub9c8\uce74\uc138 \uac15\ub0a8\uc810 (\uc77c\uc2dd) - \uc624\ub9c8\uce74\uc138 \ucf54\uc2a4\uac00 \uc778\uae30
\u2b50 3\uc704: \ub531\ud0c0\uc774 (\uc591\uc2dd) - \ud30c\uc2a4\ud0c0\uc640 \uc2a4\ud14c\uc774\ud06c

\ubaa8\ub450 \ub9db\uc788\uc73c\ub2c8 \uaf2d \uac00\ubcf4\uc138\uc694!
"""

# def extract_restaurants(text: str) -> list[dict]:
#     TODO: 정\uaddc\uc2dd\uc73c\ub85c \ub9e4\uc7a5\uba85, \uc21c\uc704, \uce74\ud14c\uace0\ub9ac\ub97c \ucd94\ucd9c\ud558\uc138\uc694
#     return []

# result = extract_restaurants(new_ai_response)
# for r in result:
#     print(r)

---
## 핵심 정리

| 개념 | 설명 | 활용 장면 |
|------|------|----------|
| JSON Mode | API에서 JSON 출력 강제 | 안정적인 파싱 필요 시 |
| Function Calling | 함수 스키마로 구조화 | 복잡한 구조 추출 |
| Pydantic | 타입/값 검증 자동화 | 프로덕션 데이터 파이프라인 |
| regex 파싱 | 텍스트에서 패턴 추출 | JSON 없는 응답 대응 |
| Retry 로직 | 파싱 실패 시 재시도 | 안정성 향상 |
| Rule vs LLM | 비용/속도 vs 유연성 | 상황에 맞는 선택 |

**다음 노트북**: [03-multi-llm-orchestration.ipynb](03-multi-llm-orchestration.ipynb) - 멀티 LLM 오케스트레이션